<a href="https://colab.research.google.com/github/PingPhantom/Colab/blob/main/nix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import asyncio, fnmatch, glob, html as html_lib, importlib, io, json, os, re, shutil, site, subprocess, sys, tarfile, time, zipfile
from pathlib import Path
from typing import Callable, Literal, TypeVar
from IPython.core.getipython import get_ipython
from IPython.core.magic import register_cell_magic
from IPython.display import display

# ==========================================
# 0. ตั้งค่าพอร์ต 18080, Draft Catalog และ MODEL_CATALOG ครบ 10 โมเดล
# ==========================================
S = chr(36)
DEFAULT_PORT: int = 18080
NVIDIA_BIN_DIRS = "/usr/local/nvidia/bin:/usr/local/cuda/bin"
NVIDIA_LIB_DIRS = "/usr/lib64-nvidia:/usr/local/nvidia/lib64:/usr/local/cuda/lib64"
os.environ["LD_LIBRARY_PATH"] = NVIDIA_LIB_DIRS

ModelOptionType = Literal[
    "none", "all", "sakura-minimax-h3", "qwen38-27b-cyber",
    "qwen38-27b-rvn-heretic", "qwen38-27b-coletti", "muse-glimmer-30b-heretic",
    "minicpm-v47-35b", "lfm25-24b-a2b", "qwen35-9b-defiant-fable",
    "ornith-15-9b", "nemotron-35-bolt-74b", "custom",
]
ALL_PRESET_KEYS: list[str] = [
    "sakura-minimax-h3", "qwen38-27b-cyber", "qwen38-27b-rvn-heretic",
    "qwen38-27b-coletti", "muse-glimmer-30b-heretic", "minicpm-v47-35b",
    "lfm25-24b-a2b", "qwen35-9b-defiant-fable", "ornith-15-9b",
    "nemotron-35-bolt-74b",
]

MODEL_OPTION: ModelOptionType = "none"
MODEL_DIR: str = "/content/models"
DRAFT_DIR: str = "/content/models/_drafts"
CUSTOM_HF_REPO: str = "AtomicChat/NVIDIA-Nemotron-3.5-Bolt-74B-A9B-Uncensored-GGUF"
CUSTOM_INCLUDE: list[str] | None = None
LAST_MODEL_DIR: str | None = None

SPEC_STATE: dict = {
    "mode": "off", "draft_max": 0, "draft_model": "",
    "kv_q8": False, "active_label": "⏹️ ปิด Speculative",
}

DRAFT_CATALOG: dict[str, dict] = {
    "qwen38-coletti-mtp": {
        "repo_id": "JonathanColetti/Qwen3.8-27B-Uncensored-GGUF",
        "include": ["*draft*Q8_0*", "*draft*Q4*"], "exclude": ["*mmproj*"],
        "is_mtp": True, "desc": "🚀 Qwen3.8-27B MTP Draft (Coletti)",
    },
    "qwen35-0.8b": {
        "repo_id": "bartowski/Qwen_Qwen3.5-0.8B-GGUF",
        "include": ["*Q4_K_M*.gguf"], "exclude": ["*mmproj*"],
        "is_mtp": False, "desc": "⚡ Qwen3.5-0.8B Q4_K_M (~0.5GB • Bartowski)",
    },
    "qwen3-0.6b": {
        "repo_id": "unsloth/Qwen3-0.6B-GGUF",
        "include": ["*Q4_K_M*.gguf"], "exclude": ["*mmproj*"],
        "is_mtp": False, "desc": "⚡ Qwen3-0.6B Q4_K_M (~0.4GB • Unsloth)",
    },
    "qwen3-coder-0.8b": {
        "repo_id": "DavidAU/Qwen3-Zero-Coder-Reasoning-V2-0.8B-NEO-EX-GGUF",
        "include": ["*Q4_K_M*.gguf", "*IQ4_XS*.gguf"], "exclude": ["*mmproj*"],
        "is_mtp": False, "desc": "💻 Qwen3-Zero-Coder-0.8B (DavidAU)",
    },
}

MODEL_CATALOG: dict[str, dict] = {
    "sakura-minimax-h3": {"repo_id": "webmp3/Sakura-MiniMax-H3-Uncensored-GGUF", "include": ["*Q4_K*", "*IQ4*"], "exclude": None, "approx_gb": 15.5, "has_mtp": False, "desc": "🌸 Sakura-MiniMax-H3-Uncensored (26B) Multimodal"},
    "qwen38-27b-cyber": {"repo_id": "cyjin-yl/Qwen3.8-27B-Uncensored-Cyber-agentic-imatrix-GGUF", "include": ["*IQ4_XS*", "*mmproj*"], "exclude": None, "approx_gb": 15.2, "has_mtp": False, "desc": "💻 Qwen3.8-27B-Uncensored-Cyber Coding & Agent"},
    "qwen38-27b-rvn-heretic": {"repo_id": "0bserverx/Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF", "include": ["RVN-Q4_K_M.gguf", "*mmproj*"], "exclude": ["*mtp*", "*multilingual*"], "approx_gb": 16.5, "has_mtp": False, "desc": "🔥 Qwen3.8-27B-RVN-Heretic (RVN Q4_K_M) ARA 3-pass"},
    "qwen38-27b-coletti": {"repo_id": "JonathanColetti/Qwen3.8-27B-Uncensored-GGUF", "include": ["*IQ4_XS*", "*mmproj*"], "exclude": ["*noMTP*", "*draft*"], "approx_gb": 15.3, "has_mtp": True, "desc": "🧠 Qwen3.8-27B-Uncensored Coletti (IQ4_XS + MTP)"},
    "muse-glimmer-30b-heretic": {"repo_id": "0bserverx/Muse-Glimmer-30B-Heretic-Uncensored-GGUF", "include": ["*IQ4_XS*", "*mmproj*"], "exclude": None, "approx_gb": 15.3, "has_mtp": False, "desc": "✨ Muse-Glimmer-30B-Heretic (IQ4_XS) Multimodal"},
    "minicpm-v47-35b": {"repo_id": "AtomicChat/MiniCPM-V-4.7-35B-A3B-Uncensored-GGUF", "include": ["*IQ4_XS*", "*Q4_K_M*", "*mmproj*"], "exclude": None, "approx_gb": 20.5, "has_mtp": False, "desc": "👁️ MiniCPM-V-4.7-35B-A3B (3B active MoE) Vision"},
    "lfm25-24b-a2b": {"repo_id": "AtomicChat/LFM2.5-24B-A2B-Uncensored-GGUF", "include": ["*Q4_K_M*", "*IQ4_XS*"], "exclude": None, "approx_gb": 14.5, "has_mtp": False, "desc": "⚡ LFM2.5-24B-A2B (2B active MoE) เร็วสูง หลายภาษา"},
    "qwen35-9b-defiant-fable": {"repo_id": "DavidAU/Qwen3.5-9B-The-Defiant-Fable-Uncensored-Heretic-NEO-IMATRIX-MAX-MTP-GGUF", "include": ["*Q4_K_M*", "*mmproj-F16*"], "exclude": ["*draft*"], "approx_gb": 6.4, "has_mtp": True, "desc": "📖 Qwen3.5-9B-Defiant-Fable (Q4_K_M + MTP) T4 ลื่น"},
    "ornith-15-9b": {"repo_id": "mradermacher/Ornith-1.5-9B-uncensored-GGUF", "include": ["*Q4_K_M*", "*mmproj-Q8_0*"], "exclude": None, "approx_gb": 6.4, "has_mtp": False, "desc": "🦅 Ornith-1.5-9B-uncensored (Q4_K_M) ลง T4 ได้ทั้งตัว"},
    "nemotron-35-bolt-74b": {"repo_id": "AtomicChat/NVIDIA-Nemotron-3.5-Bolt-74B-A9B-Uncensored-GGUF", "include": ["*IQ2*", "*IQ3_XXS*", "*mmproj*"], "exclude": None, "approx_gb": 24.5, "has_mtp": False, "desc": "🚀 Nemotron-3.5-Bolt-74B-A9B (9B active MoE)"},
    "custom": {"repo_id": CUSTOM_HF_REPO, "include": CUSTOM_INCLUDE, "exclude": None, "approx_gb": 0.0, "has_mtp": False, "desc": "⚙️ กำหนด Repo เอง"},
}

# ==========================================
# 1. ติดตั้ง Nix + flake.nix + uv (.venv พร้อม pypdf)
# ==========================================
def run_quiet(cmd: str):
    p = subprocess.run(["/bin/bash", "-c", cmd], capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout + p.stderr, file=sys.stderr)
        raise subprocess.CalledProcessError(p.returncode, cmd)

def run_stream(cmd: str):
    p = subprocess.Popen(["/bin/bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    if p.stdout:
        for line in p.stdout:
            print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

Path("/etc/nix").mkdir(parents=True, exist_ok=True)
Path("/etc/nix/nix.conf").write_text("experimental-features = nix-command flakes\nsandbox = false\nfilter-syscalls = false\nbuild-users-group =\nmax-jobs = auto\ncores = 0\nhttp-connections = 50\n", encoding="utf-8")
for p in reversed([str(Path.home() / ".local/bin"), str(Path.home() / ".nix-profile/bin"), "/nix/var/nix/profiles/default/bin", "/usr/local/nvidia/bin", "/usr/local/cuda/bin"]):
    if p not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{p}{os.pathsep}{os.environ.get('PATH', '')}"

if shutil.which("nix") is None:
    print("⏳ กำลังติดตั้ง Nix Package Manager...")
    run_quiet("groupadd -r nixbld 2>/dev/null || true && curl -sL https://nixos.org/nix/install | sh -s -- --no-daemon")

dl_ex = shutil.which("download-deepseek")
if dl_ex and ("\\" + S) in Path(dl_ex).read_text(encoding="utf-8", errors="ignore"):
    Path(dl_ex).unlink(missing_ok=True)

if shutil.which("download-deepseek") is None:
    Path("default.nix").unlink(missing_ok=True)
    py_ver = f"python{sys.version_info.major}{sys.version_info.minor}"
    Path("flake.nix").write_text("\n".join([
        "{ outputs = { self, nixpkgs }:",
        f'  let system = "x86_64-linux"; pkgs = nixpkgs.legacyPackages.{S}{{system}}; pyPkg = pkgs.{py_ver} or pkgs.python3;',
        f'      hfRepoId = "{CUSTOM_HF_REPO}";',
        '      downloadDeepSeek = pkgs.writeShellScriptBin "download-deepseek" \'\'',
        f'        set -e; REPO="\'\'{S}{{HF_REPO_ID:-{S}{{hfRepoId}}}}"',
        f'        if [[ "\'\'{S}{{1:-}}" == -* ]] || [ -z "\'\'{S}{{1:-}}" ]; then MODEL_DIR="/content/models"; else MODEL_DIR="{S}1"; shift; fi',
        f'        mkdir -p "{S}MODEL_DIR"',
        f'        {S}{{pkgs.python3Packages.huggingface-hub}}/bin/hf download "{S}REPO" --local-dir "{S}MODEL_DIR" "{S}@"',
        f'        echo "✅ ดาวน์โหลดเสร็จที่: {S}MODEL_DIR"',
        "      '';",
        f"  in {{ devShells.{S}{{system}}.default = pkgs.mkShell {{",
        "    packages = [ pyPkg pkgs.python3Packages.huggingface-hub pkgs.uv pkgs.ruff pkgs.jq pkgs.ripgrep pkgs.curl pkgs.gnused pkgs.aria2 downloadDeepSeek ];",
        f'    env = {{ UV_PYTHON_DOWNLOADS = "never"; UV_PYTHON = "{S}{{pyPkg}}/bin/python"; LD_LIBRARY_PATH = pkgs.lib.makeLibraryPath [ pkgs.stdenv.cc.cc.lib pkgs.zlib pkgs.glib ]; }};',
        "  }; }; }",
    ]) + "\n", encoding="utf-8")
    print("⏳ กำลังโหลดสภาพแวดล้อม Nix DevShell...")
    run_quiet("nix registry pin nixpkgs github:NixOS/nixpkgs/nixos-unstable")
    env_json = subprocess.check_output(["nix", "develop", "--command", "python3", "-c", "import os, json; print(json.dumps(dict(os.environ)))"], text=True, stderr=subprocess.DEVNULL)
    for k, v in json.loads(env_json).items():
        if k not in ("_", "SHLVL", "PWD", "OLDPWD", "PATH", "LD_LIBRARY_PATH"):
            os.environ[k] = v
        elif k == "PATH":
            os.environ["PATH"] = f"/usr/bin:/bin:{v}:{NVIDIA_BIN_DIRS}:/usr/local/sbin:/usr/local/bin:/usr/sbin:/sbin"
        elif k == "LD_LIBRARY_PATH":
            os.environ["LD_LIBRARY_PATH"] = f"{v}:{NVIDIA_LIB_DIRS}"
    for b_nm in ("bash", "sh"):
        (Path("/usr/local/bin") / b_nm).unlink(missing_ok=True)
    for tool in ("uv", "ruff", "jq", "rg", "hf", "download-deepseek"):
        tp = shutil.which(tool)
        if tp:
            t = Path("/usr/local/bin") / tool; t.unlink(missing_ok=True); t.symlink_to(tp)

if not Path(".venv").exists():
    print("⏳ กำลังเตรียมแพ็กเกจ Python (uv)...")
    run_stream("set -e\nif [ ! -f pyproject.toml ]; then uv init --vcs none --no-readme --quiet; fi\nuv add --quiet numpy requests rich openai fastembed ipywidgets pydantic pypdf\nuv export --format requirements-txt --no-hashes --no-emit-project --quiet -o requirements.txt\n")

pp = Path("pyproject.toml")
if "[tool.ruff]" not in pp.read_text(encoding="utf-8"):
    pp.write_text(pp.read_text(encoding="utf-8").rstrip() + f'\n\n[tool.ruff]\nline-length = 88\ntarget-version = "py{sys.version_info.major}{sys.version_info.minor}"\n[tool.ruff.lint]\nselect = ["E","F","W","I","UP","B","SIM"]\nignore = ["E501"]\n', encoding="utf-8")

venv_path = Path(".venv").resolve()
venv_site = next(iter(glob.glob(str(venv_path / "lib" / "python*" / "site-packages"))))
site.addsitedir(venv_site)
if venv_site in sys.path:
    sys.path.remove(venv_site)
sys.path.insert(0, venv_site)
os.environ["VIRTUAL_ENV"] = str(venv_path)
os.environ["PATH"] = f"{venv_path / 'bin'}{os.pathsep}{os.environ.get('PATH', '')}"
importlib.invalidate_caches()

# ==========================================
# 2. ฟังก์ชันจัดการดิสก์, GPU, โมเดลหลัก + Draft & llama-server
# ==========================================
import ipywidgets as widgets
import numpy as np
import requests
from fastembed import TextEmbedding
from openai import AsyncOpenAI, OpenAI
from pydantic import BaseModel
from pypdf import PdfReader
from rich import print as rprint
from rich.console import Console
from rich.syntax import Syntax
from rich.table import Table

T = TypeVar("T", bound=BaseModel)
LLAMA_SERVER_PROC: subprocess.Popen | None = None
LLAMA_LOG_FILE = Path("/content/llama_server.log")

def uv_add(*packages: str):
    s = " ".join(packages)
    run_quiet(f"uv add --quiet {s} && uv export --format requirements-txt --no-hashes --no-emit-project --quiet -o requirements.txt")
    importlib.invalidate_caches()
    print(f"✅ ติดตั้ง {s} เรียบร้อย!")

def _dir_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return path.stat().st_size if path.is_file() else sum(p.stat().st_size for p in path.rglob("*") if not p.is_symlink() and p.is_file())

def _fmt_bytes(n: int) -> str:
    return f"{n / (1024**3):.2f} GB" if n >= 1024**3 else (f"{n / (1024**2):.2f} MB" if n >= 1024**2 else f"{n / 1024:.1f} KB")

def check_disk_space(target_dir: str = "/content", required_gb: float | None = None) -> dict:
    cp = Path(target_dir)
    while not cp.exists() and cp != cp.parent:
        cp = cp.parent
    u = shutil.disk_usage(cp)
    t_gb, u_gb, f_gb = u.total / (1024**3), u.used / (1024**3), u.free / (1024**3)
    col = "green" if (required_gb is None or f_gb >= required_gb) else "bold red"
    extra = "" if required_gb is None else (f" — [bold green]✅ พอ ({required_gb:.1f}GB)[/bold green]" if f_gb >= required_gb else f" — [bold red]❌ ขาด {required_gb - f_gb:.1f}GB[/bold red]")
    Console().print(f"💾 [bold]Disk ({cp}):[/bold] ว่าง [{col}]{f_gb:.2f} GB[/{col}] / {t_gb:.2f} GB{extra}")
    return {"total_gb": t_gb, "used_gb": u_gb, "free_gb": f_gb}

def check_gpu_and_recommend(gb_per_layer: float = 0.5, kv_headroom_gb: float = 2.0, verbose: bool = True) -> dict:
    cores = os.cpu_count() or 2
    threads = max(1, cores - 1) if cores > 2 else cores
    ram_t, ram_a = 0.0, 0.0
    if Path("/proc/meminfo").exists():
        for ln in Path("/proc/meminfo").read_text().splitlines():
            pts = ln.split()
            if len(pts) >= 2 and ln.startswith("MemTotal:"):
                _, kb, *_ = pts; ram_t = int(kb) / (1024**2)
            elif len(pts) >= 2 and ln.startswith("MemAvailable:"):
                _, kb, *_ = pts; ram_a = int(kb) / (1024**2)
    smi = next((c for c in ("/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi", shutil.which("nvidia-smi")) if c and Path(c).exists()), None)
    gpus = []
    if smi:
        try:
            out = subprocess.check_output([smi, "--query-gpu=index,name,memory.total,memory.free,compute_cap", "--format=csv,noheader,nounits"], text=True, stderr=subprocess.DEVNULL).strip()
            for ln in out.splitlines():
                if ln.strip():
                    i, nm, tot, fr, cc = [x.strip() for x in ln.split(",")]
                    gpus.append({"index": int(i), "name": nm, "vram_total_gb": float(tot) / 1024, "vram_free_gb": float(fr) / 1024, "compute_cap": cc})
        except Exception:
            pass
    if not gpus:
        return {"has_gpu": False, "gpu_name": "CPU Only", "vram_total_gb": 0.0, "vram_free_gb": 0.0, "ram_total_gb": ram_t, "ram_avail_gb": ram_a, "recommended_gpu_layers": 0, "recommended_ctx_size": 4096, "recommended_threads": threads}
    tv, fv = sum(g["vram_total_gb"] for g in gpus), sum(g["vram_free_gb"] for g in gpus)
    names = ", ".join(f"{g['name']} ({g['vram_total_gb']:.0f}GB)" for g in gpus)
    layers = max(1, int(max(0.0, fv - kv_headroom_gb) / gb_per_layer))
    ctx = 16384 if tv >= 35 else (8192 if tv >= 20 else 4096)
    if verbose:
        Console().print(f"🎮 [bold green]GPU:[/bold green] {names} | VRAM ว่าง {fv:.2f}/{tv:.2f} GB | RAM ว่าง {ram_a:.1f} GB")
    return {"has_gpu": True, "gpu_name": names, "vram_total_gb": tv, "vram_free_gb": fv, "ram_total_gb": ram_t, "ram_avail_gb": ram_a, "recommended_gpu_layers": layers, "recommended_ctx_size": ctx, "recommended_threads": threads}

def _download_and_extract(url: str, name: str, dest: Path):
    arc = dest / name
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(arc, "wb") as f:
            for ch in r.iter_content(chunk_size=1024 * 1024):
                if ch:
                    f.write(ch)
    if name.endswith(".tar.gz"):
        with tarfile.open(arc, "r:gz") as tf:
            tf.extractall(dest)
    elif name.endswith(".zip"):
        with zipfile.ZipFile(arc, "r") as zf:
            zf.extractall(dest)
    arc.unlink(missing_ok=True)

def install_llama_server(install_dir: str = "/content/.llama-cpp") -> str:
    wrapper = Path("/usr/local/bin/llama-server")
    if wrapper.exists():
        wtxt = wrapper.read_text(encoding="utf-8", errors="ignore")
        if ("\\" + S) in wtxt:
            wrapper.write_text(wtxt.replace("\\" + S, S), encoding="utf-8"); wrapper.chmod(0o755)
        if subprocess.run([str(wrapper), "--version"], capture_output=True).returncode == 0:
            return str(wrapper)
    rprint("[bold cyan]⏳ กำลังติดตั้ง Prebuilt llama-server (CUDA 12)...[/bold cyan]")
    hw = check_gpu_and_recommend(verbose=False)
    rels = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10", timeout=20).json()
    main_asset, cudart_asset = None, None
    prefs = ["bin-ubuntu-cuda-12", "bin-ubuntu-vulkan-x64", "bin-ubuntu-x64"] if hw["has_gpu"] else ["bin-ubuntu-x64"]
    for pat in prefs:
        for rel in rels:
            assets = rel.get("assets", [])
            for a in assets:
                nm = a.get("name", "")
                if nm.startswith("llama-") and pat in nm and "x64" in nm and (nm.endswith(".tar.gz") or nm.endswith(".zip")):
                    main_asset = a
                    if "cuda-12" in pat:
                        cudart_asset = next((c for c in assets if c.get("name", "").startswith("cudart-") and "ubuntu-cuda-12" in c.get("name", "") and "x64" in c.get("name", "")), None)
                    break
            if main_asset:
                break
        if main_asset:
            break
    if not main_asset:
        raise RuntimeError("ไม่พบไฟล์ Prebuilt llama-server")
    dest = Path(install_dir); shutil.rmtree(dest, ignore_errors=True); dest.mkdir(parents=True, exist_ok=True)
    if cudart_asset:
        _download_and_extract(cudart_asset["browser_download_url"], cudart_asset["name"], dest)
    _download_and_extract(main_asset["browser_download_url"], main_asset["name"], dest)
    srv_bins = [p for p in dest.rglob("llama-server") if p.is_file()]
    if not srv_bins:
        raise RuntimeError("ไม่พบไบนารี llama-server")
    srv_bin = next(iter(srv_bins)); srv_bin.chmod(0o755)
    lib_dirs = ":".join(sorted({str(p.parent) for p in dest.rglob("*.so*")}))
    wrapper.unlink(missing_ok=True)
    wrapper.write_text(f'#!/usr/bin/env bash\nexport LD_LIBRARY_PATH="{NVIDIA_LIB_DIRS}:{lib_dirs}:{srv_bin.parent}:{S}{{LD_LIBRARY_PATH:-}}"\nexec "{srv_bin}" "{S}@"\n', encoding="utf-8")
    wrapper.chmod(0o755)
    rprint(f"[bold green]✅ ติดตั้ง llama-server สำเร็จ ({main_asset['name']})![/bold green]")
    return str(wrapper)

def list_model_files(repo_id: str = CUSTOM_HF_REPO):
    r = requests.get(f"https://huggingface.co/api/models/{repo_id}/tree/main?recursive=true", timeout=15); r.raise_for_status()
    items = [x for x in r.json() if x.get("type") == "file"]
    tb = Table(title=f"📦 ไฟล์ใน {repo_id}")
    tb.add_column("#", justify="right"); tb.add_column("Filename", style="cyan"); tb.add_column("Size", justify="right", style="green")
    for i, it in enumerate(items, 1):
        tb.add_row(str(i), it["path"], _fmt_bytes(it.get("size", 0)))
    Console().print(tb)
    return items

def download_model_files(include: list[str] | str | None = None, exclude: list[str] | str | None = None, model_dir: str = MODEL_DIR, repo_id: str = CUSTOM_HF_REPO, force: bool = False):
    inc = [include] if isinstance(include, str) else include
    exc = [exclude] if isinstance(exclude, str) else exclude
    try:
        r = requests.get(f"https://huggingface.co/api/models/{repo_id}/tree/main?recursive=true", timeout=15); r.raise_for_status()
        need = 0
        for it in (x for x in r.json() if x.get("type") == "file"):
            fn = it["path"]
            if inc and not any(fnmatch.fnmatch(fn, p) or fnmatch.fnmatch(Path(fn).name, p) for p in inc):
                continue
            if exc and any(fnmatch.fnmatch(fn, p) or fnmatch.fnmatch(Path(fn).name, p) for p in exc):
                continue
            lf = Path(model_dir) / fn
            if lf.exists() and lf.stat().st_size == it.get("size", 0):
                continue
            need += it.get("size", 0)
        req_gb = need / (1024**3)
        di = check_disk_space(model_dir, required_gb=req_gb)
        if di["free_gb"] < req_gb and not force:
            raise RuntimeError(f"ดิสก์ไม่พอ: ต้องการ {req_gb:.2f} GB แต่ว่าง {di['free_gb']:.2f} GB")
    except requests.RequestException:
        check_disk_space(model_dir)
    args = [f'"{model_dir}"'] + [f'--include "{p}"' for p in (inc or [])] + [f'--exclude "{p}"' for p in (exc or [])]
    os.environ["HF_REPO_ID"] = repo_id
    run_stream("download-deepseek " + " ".join(args))

def _find_draft_gguf(folder: Path) -> Path | None:
    if not folder.is_dir():
        return None
    cands = sorted([p for p in folder.rglob("*.gguf") if "mmproj" not in p.name.lower() and "imatrix" not in p.name.lower()], key=lambda p: p.stat().st_size)
    return next(iter(cands), None)

def install_draft_model(draft_key_or_spec: str, draft_base_dir: str = DRAFT_DIR) -> tuple[str, bool]:
    base = Path(draft_base_dir)
    if draft_key_or_spec in DRAFT_CATALOG:
        cfg = DRAFT_CATALOG[draft_key_or_spec]; sub = base / draft_key_or_spec
        rprint(f"[bold magenta]⚡ กำลังโหลดไฟล์ Draft:[/bold magenta] [cyan]{draft_key_or_spec}[/cyan] -> {cfg['repo_id']}")
        download_model_files(include=cfg["include"], exclude=cfg["exclude"], model_dir=str(sub), repo_id=cfg["repo_id"])
        found = _find_draft_gguf(sub)
        if not found:
            raise FileNotFoundError(f"ไม่พบไฟล์ Draft .gguf ใน {sub}")
        rprint(f"[bold green]✅ พร้อมใช้ Draft:[/bold green] {found}")
        return str(found), bool(cfg.get("is_mtp", False))
    spec = draft_key_or_spec.strip()
    if Path(spec).is_file():
        return spec, ("mtp" in Path(spec).name.lower())
    repo_part, pat_part = spec.split(":", 1) if ":" in spec else (spec, "*Q4_K_M*.gguf")
    repo_part, pat_part = repo_part.strip(), pat_part.strip() or "*Q4_K_M*.gguf"
    if "/" not in repo_part:
        raise ValueError("ระบุเป็น org/repo หรือ org/repo:*Q4_K_M*.gguf หรือพาธไฟล์ .gguf")
    sub = base / ("custom_" + re.sub(r"[^a-zA-Z0-9_.-]+", "_", repo_part))
    download_model_files(include=[pat_part], exclude=["*mmproj*"], model_dir=str(sub), repo_id=repo_part)
    found = _find_draft_gguf(sub)
    if not found:
        raise FileNotFoundError(f"ไม่พบไฟล์ .gguf ({pat_part}) ใน {repo_part}")
    return str(found), ("mtp" in found.name.lower())

def install_model(option: ModelOptionType = "qwen38-27b-cyber", model_dir: str = MODEL_DIR, repo_id: str | None = None, include: list[str] | str | None = None, exclude: list[str] | str | None = None, auto_start_server: bool = False, port: int = DEFAULT_PORT, force: bool = False) -> dict | list[dict]:
    if option == "all":
        return install_all_models(model_dir=model_dir, force=force, port=port)
    if option not in MODEL_CATALOG:
        raise ValueError(f"เลือกโมเดลจาก: {[*MODEL_CATALOG.keys(), 'all']}")
    cfg = MODEL_CATALOG[option]; sub = str(Path(model_dir) / option)
    rprint(f"[bold magenta]📦 เตรียมโมเดล:[/bold magenta] [cyan]{option}[/cyan] -> {repo_id or cfg['repo_id']}")
    download_model_files(include=include if include is not None else cfg["include"], exclude=exclude if exclude is not None else cfg["exclude"], model_dir=sub, repo_id=repo_id or cfg["repo_id"], force=force)
    if auto_start_server:
        start_strata_server(model_dir=sub, port=port)
    return {"option": option, "model_dir": sub}

def install_all_models(options: list[str] | None = None, model_dir: str = MODEL_DIR, force: bool = False, start_after: str | None = None, port: int = DEFAULT_PORT) -> list[dict]:
    keys = options or ALL_PRESET_KEYS
    est = sum(MODEL_CATALOG[k]["approx_gb"] for k in keys if k in MODEL_CATALOG)
    di = check_disk_space(model_dir, required_gb=est)
    if di["free_gb"] < est and not force:
        raise RuntimeError(f"พื้นที่ไม่พอโหลดครบ {len(keys)} โมเดล (~{est:.1f} GB)")
    res = []
    for i, k in enumerate(keys, 1):
        rprint(f"\n[bold cyan]📥 [{i}/{len(keys)}] ดาวน์โหลด {k}[/bold cyan]")
        try:
            info = install_model(k, model_dir=model_dir, force=force, port=port)  # type: ignore[arg-type]
            if isinstance(info, dict):
                res.append({**info, "status": "success"})
        except Exception as e:
            rprint(f"[bold red]❌ {k} ล้มเหลว: {e}[/bold red]"); res.append({"option": k, "status": f"failed: {e}"})
    if start_after and start_after in MODEL_CATALOG:
        switch_strata_model(start_after, model_dir=model_dir, port=port)
    return res

def clean_download_cache(model_dir: str = MODEL_DIR, include_global_hf_cache: bool = True) -> float:
    freed, base = 0, Path(model_dir)
    if base.exists():
        for cd in base.rglob(".cache"):
            if cd.is_dir():
                freed += _dir_bytes(cd); shutil.rmtree(cd, ignore_errors=True)
        for pat in ("*.incomplete", "*.part", "*.tmp"):
            for tf in base.rglob(pat):
                if tf.is_file():
                    freed += tf.stat().st_size; tf.unlink(missing_ok=True)
    if include_global_hf_cache:
        hc = Path.home() / ".cache/huggingface/hub"
        if hc.exists():
            freed += _dir_bytes(hc); shutil.rmtree(hc, ignore_errors=True)
    rprint(f"[bold green]🧹 ล้างแคชสำเร็จ คืนพื้นที่ {_fmt_bytes(freed)}[/bold green]")
    return round(freed / (1024**3), 2)

def remove_model(options: str | list[str], model_dir: str = MODEL_DIR, clear_cache: bool = True, stop_server: bool = False) -> dict:
    base = Path(model_dir)
    if not base.exists():
        return {"removed": [], "freed_gb": 0.0}
    if stop_server or options == "all":
        stop_strata_server()
    targets = [d.name for d in base.iterdir() if d.is_dir()] if options == "all" else ([options] if isinstance(options, str) else list(options))
    before = shutil.disk_usage(base).free; removed = []
    for opt in targets:
        tp = base / opt
        if tp.is_dir():
            sz = _dir_bytes(tp); shutil.rmtree(tp, ignore_errors=True); removed.append(opt)
            rprint(f"[bold yellow]🗑️ ลบ '{opt}' ({_fmt_bytes(sz)})[/bold yellow]")
    if clear_cache:
        clean_download_cache(model_dir=model_dir, include_global_hf_cache=False)
    freed = max(0, shutil.disk_usage(base if base.exists() else Path("/content")).free - before)
    rprint(f"[bold green]✅ คืนพื้นที่ดิสก์รวม {_fmt_bytes(freed)}[/bold green]")
    return {"removed": removed, "freed_gb": round(freed / (1024**3), 2)}

def keep_only_models(keep: str | list[str], model_dir: str = MODEL_DIR) -> dict:
    base = Path(model_dir)
    if not base.exists():
        return {"kept": [], "removed": [], "freed_gb": 0.0}
    ks = {keep} if isinstance(keep, str) else set(keep)
    rem = [d.name for d in base.iterdir() if d.is_dir() and not d.name.startswith("_") and d.name not in ks]
    return remove_model(rem, model_dir=model_dir) if rem else {"kept": sorted(ks), "removed": [], "freed_gb": 0.0}

def get_openai_client(port: int = DEFAULT_PORT, api_key: str = "strata", async_mode: bool = False, timeout: float = 300.0) -> OpenAI | AsyncOpenAI:
    url = f"http://127.0.0.1:{port}/v1"
    return AsyncOpenAI(base_url=url, api_key=api_key, timeout=timeout) if async_mode else OpenAI(base_url=url, api_key=api_key, timeout=timeout)

def _check_server_online(port: int = DEFAULT_PORT) -> tuple[bool, str]:
    try:
        if requests.get(f"http://127.0.0.1:{port}/health", timeout=1.5).status_code != 200:
            return False, "loading"
        rm = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=1.5)
        if rm.status_code == 200 and "json" in rm.headers.get("content-type", "").lower():
            d = rm.json()
            if isinstance(d, dict) and isinstance(d.get("data"), list) and d["data"]:
                return True, str(next(iter(d["data"])).get("id", "default"))
    except Exception:
        pass
    return False, "offline"

def _pick_main_gguf(model_dir: str) -> tuple[Path, Path | None, float]:
    all_ggufs = [p for p in sorted(Path(model_dir).rglob("*.gguf")) if "_drafts" not in p.parts]
    if not all_ggufs:
        raise FileNotFoundError(f"ไม่พบไฟล์ .gguf ใน {model_dir}")
    mmprojs = [p for p in all_ggufs if "mmproj" in p.name.lower()]
    mains = [p for p in all_ggufs if "mmproj" not in p.name.lower() and "imatrix" not in p.name.lower() and "-draft" not in p.name.lower() and "draft-" not in p.name.lower()]
    if not mains:
        raise FileNotFoundError(f"ไม่พบไฟล์โมเดลหลักใน {model_dir}")
    cands: list[tuple[float, Path]] = []
    s_re = re.compile(r"^(.*)-00001-of-(\d{5})\.gguf\Z", re.IGNORECASE)
    for p in mains:
        m = s_re.match(p.name)
        if m:
            pfx, tot_s = m.groups()
            shards = [p.parent / f"{pfx}-{i:05d}-of-{tot_s}.gguf" for i in range(1, int(tot_s) + 1)]
            if all(s.exists() for s in shards):
                cands.append((sum(s.stat().st_size for s in shards) / (1024**3), p))
        elif "-of-" not in p.name.lower():
            cands.append((p.stat().st_size / (1024**3), p))
    mm_pick = next(iter(mmprojs), None)
    if not cands:
        m0 = next(iter(mains)); return m0, mm_pick, m0.stat().st_size / (1024**3)
    cands.sort(key=lambda x: x[0])
    sz, chosen = next(iter(cands))
    return chosen, mm_pick, sz

def stop_strata_server():
    global LLAMA_SERVER_PROC
    if LLAMA_SERVER_PROC and LLAMA_SERVER_PROC.poll() is None:
        LLAMA_SERVER_PROC.terminate()
        try:
            LLAMA_SERVER_PROC.wait(timeout=8)
        except subprocess.TimeoutExpired:
            LLAMA_SERVER_PROC.kill()
    LLAMA_SERVER_PROC = None
    subprocess.run(["/bin/bash", "-c", "pkill -f llama-server || true"])
    SPEC_STATE["active_label"] = "⏹️ ปิด Speculative"
    rprint("[bold yellow]🛑 หยุดการทำงานของ API Server เรียบร้อย[/bold yellow]")

def strata_server_status(port: int = DEFAULT_PORT):
    on, mid = _check_server_online(port)
    rprint(f"[bold {'green' if on else 'red'}]{'🟢 Online' if on else '🔴 Offline'} (:{port} • {mid} • {SPEC_STATE['active_label']})[/bold {'green' if on else 'red'}]")
    if LLAMA_LOG_FILE.exists():
        print("\n".join(LLAMA_LOG_FILE.read_text(encoding="utf-8", errors="ignore").splitlines()[-15:]))

def start_strata_server(model_dir: str | None = MODEL_DIR, port: int = DEFAULT_PORT, ctx_size: int = 4096, restart: bool = True, spec_mode: str | None = None, draft_max: int | None = None, draft_model: str | None = None, kv_q8: bool | None = None, extra_args: list[str] | None = None, **_) -> OpenAI:
    global LLAMA_SERVER_PROC, LAST_MODEL_DIR
    if not model_dir:
        raise ValueError("กรุณาระบุ model_dir")
    srv_bin = install_llama_server()
    if restart:
        stop_strata_server()
    main_gguf, mmproj_gguf, size_gb = _pick_main_gguf(model_dir)
    LAST_MODEL_DIR = str(model_dir)
    hw = check_gpu_and_recommend(verbose=False)
    help_txt = subprocess.run([srv_bin, "--help"], capture_output=True, text=True).stdout
    alias_name = Path(model_dir).name

    smode = (spec_mode if spec_mode is not None else str(SPEC_STATE["mode"])).lower().strip()
    dmax_in = int(draft_max if draft_max is not None else SPEC_STATE["draft_max"])
    dmodel_in = (draft_model if draft_model is not None else str(SPEC_STATE["draft_model"])).strip()
    use_kv_q8 = bool(kv_q8 if kv_q8 is not None else SPEC_STATE["kv_q8"])
    has_draft = bool(dmodel_in and Path(dmodel_in).is_file())
    cat_mtp = bool(MODEL_CATALOG.get(alias_name, {}).get("has_mtp")) or ("mtp" in main_gguf.name.lower())

    if smode in ("auto", "draft"):
        eff_mode = "draft_file" if has_draft else ("mtp" if cat_mtp else "ngram")
    elif smode == "mtp":
        eff_mode = "draft_file_mtp" if has_draft else "mtp"
    elif smode == "ngram":
        eff_mode = "ngram"
    else:
        eff_mode = "off"

    def _build_cmd(target_eff: str) -> tuple[list[str], str]:
        cmd = [srv_bin, "--model", str(main_gguf), "--alias", alias_name, "--host", "127.0.0.1", "--port", str(port), "--ctx-size", str(ctx_size), "--threads", str(hw["recommended_threads"])]
        if hw["has_gpu"]:
            if "--fit" in help_txt:
                cmd.extend(["--fit", "on", "--fit-target", "1024"])
            else:
                usable = max(1.0, hw["vram_free_gb"] - 2.2)
                cmd.extend(["--n-gpu-layers", str(99 if size_gb <= usable else max(1, int(48 * (usable / max(size_gb, 1.0)))))])
            if "--flash-attn" in help_txt:
                cmd.extend(["--flash-attn", "on"])
        else:
            cmd.extend(["--n-gpu-layers", "0"])
        if "--jinja" in help_txt:
            cmd.append("--jinja")
        if use_kv_q8 and "--cache-type-k" in help_txt:
            cmd.extend(["--cache-type-k", "q8_0", "--cache-type-v", "q8_0"])

        dmax_flag = "--draft-max" if "--draft-max" in help_txt else "--draft"
        no_mm = False; lbl = "⏹️ ปิด Speculative"
        if target_eff == "mtp" and "--spec-type" in help_txt:
            dm = dmax_in if dmax_in > 0 else 2
            cmd.extend(["--spec-type", "draft-mtp", dmax_flag, str(dm)]); no_mm = True; lbl = f"🚀 MTP (max={dm})"
        elif target_eff == "ngram" and "--spec-type" in help_txt:
            ng = "ngram-mod" if "ngram-mod" in help_txt else ("ngram-simple" if "ngram-simple" in help_txt else "ngram-cache")
            dm = dmax_in if dmax_in > 0 else 16
            cmd.extend(["--spec-type", ng, dmax_flag, str(dm)]); lbl = f"📚 N-gram ({ng}, max={dm})"
        elif target_eff in ("draft_file", "draft_file_mtp") and has_draft:
            dm = dmax_in if dmax_in > 0 else (3 if target_eff == "draft_file_mtp" else 6)
            cmd.extend(["--model-draft", dmodel_in, "-ngld", "99", dmax_flag, str(dm)])
            if target_eff == "draft_file_mtp" and "--spec-type" in help_txt:
                cmd.extend(["--spec-type", "draft-mtp"]); no_mm = True
            lbl = f"⚡ Draft ({Path(dmodel_in).name[:16]}.., max={dm})"
        if use_kv_q8:
            lbl += " + KV:Q8"
        if no_mm:
            if "--no-mmproj" in help_txt:
                cmd.append("--no-mmproj")
        elif mmproj_gguf and hw["vram_free_gb"] >= size_gb + 3.0:
            cmd.extend(["--mmproj", str(mmproj_gguf)])
        cmd.extend(str(a) for a in (extra_args or []))
        return cmd, lbl

    def _spawn(target_eff: str) -> tuple[bool, str]:
        global LLAMA_SERVER_PROC
        cmd_run, lbl_run = _build_cmd(target_eff)
        rprint(f"[bold cyan]🚀 เปิด llama-server (:{port}):[/bold cyan] [yellow]{main_gguf.name}[/yellow] ({size_gb:.2f}GB) • [magenta]{lbl_run}[/magenta]")
        with open(LLAMA_LOG_FILE, "w", encoding="utf-8") as log_f:
            LLAMA_SERVER_PROC = subprocess.Popen(cmd_run, stdout=log_f, stderr=subprocess.STDOUT, text=True)
        for sec in range(240):
            if LLAMA_SERVER_PROC.poll() is not None:
                return False, "\n".join(LLAMA_LOG_FILE.read_text(encoding="utf-8", errors="ignore").splitlines()[-25:])
            is_ok, loaded_id = _check_server_online(port)
            if is_ok:
                SPEC_STATE["active_label"] = lbl_run
                rprint(f"[bold green]✅ Server พร้อมใน {sec + 1}s! (model='{loaded_id}' • {lbl_run})[/bold green]")
                return True, loaded_id
            if (sec + 1) % 10 == 0:
                rprint(f"[dim]   ⏳ กำลังโหลดโมเดลลง GPU/RAM... ({sec + 1}s)[/dim]")
            time.sleep(1.0)
        raise TimeoutError("โหลดโมเดลเกิน 240 วินาที")

    fallbacks = [eff_mode] + (["draft_file", "ngram"] if eff_mode == "draft_file_mtp" else (["ngram"] if eff_mode in ("mtp", "draft_file") else []))
    last_err = ""
    for idx, m_try in enumerate(fallbacks):
        if idx > 0:
            rprint(f"[bold yellow]⚠️ โหมดก่อนหน้าไม่รองรับ กำลังสลับเป็น '{m_try}' อัตโนมัติ...[/bold yellow]")
            subprocess.run(["/bin/bash", "-c", "pkill -f llama-server || true"])
        ok, info = _spawn(m_try)
        if ok:
            return get_openai_client(port=port)  # type: ignore[return-value]
        last_err = info
    raise RuntimeError(f"llama-server หยุดทำงาน:\n{last_err}")

launch_strata = start_strata_server

def switch_strata_model(option: str, model_dir: str = MODEL_DIR, port: int = DEFAULT_PORT) -> OpenAI:
    if option not in MODEL_CATALOG:
        raise ValueError(f"เลือกโมเดลจาก: {ALL_PRESET_KEYS}")
    return start_strata_server(model_dir=str(Path(model_dir) / option), port=port, restart=True)

# ==========================================
# 3. StrataRAG (อ่านไฟล์ PDF แยกตามหน้า + อัปโหลดผ่านเบราว์เซอร์)
# ==========================================
class StrataRAG:
    def __init__(self, embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.embedding_model = embedding_model
        self._embedder: TextEmbedding | None = None
        self.chunks: list[dict] = []
        self.embeddings: np.ndarray | None = None

    @property
    def embedder(self) -> TextEmbedding:
        if self._embedder is None:
            self._embedder = TextEmbedding(model_name=self.embedding_model)
        return self._embedder

    def clear(self):
        self.chunks.clear(); self.embeddings = None

    def add_documents(self, docs: list[str | dict], chunk_size: int = 500, overlap: int = 80):
        new_c = []
        for idx, doc in enumerate(docs):
            txt = doc["text"] if isinstance(doc, dict) else str(doc)
            src = doc.get("source", f"doc_{len(self.chunks) + idx + 1}") if isinstance(doc, dict) else f"doc_{len(self.chunks) + idx + 1}"
            st = 0
            while st < len(txt):
                pc = txt[st : st + chunk_size].strip()
                if pc:
                    new_c.append({"text": pc, "source": src})
                st += max(1, chunk_size - overlap)
        if not new_c:
            return
        mat = np.vstack(list(self.embedder.embed([c["text"] for c in new_c]))).astype(np.float32)
        mat /= np.linalg.norm(mat, axis=1, keepdims=True) + 1e-10
        self.chunks.extend(new_c)
        self.embeddings = mat if self.embeddings is None else np.vstack([self.embeddings, mat])
        rprint(f"[bold green]📚 เพิ่มเข้า StrataRAG สำเร็จ:[/bold green] รวม {len(self.chunks)} chunks")

    def add_pdf_bytes(self, data: bytes, filename: str = "document.pdf", chunk_size: int = 500, overlap: int = 80) -> int:
        reader = PdfReader(io.BytesIO(data))
        docs = [{"source": f"{filename} (หน้า {pno})", "text": (pg.extract_text() or "").strip()} for pno, pg in enumerate(reader.pages, 1) if (pg.extract_text() or "").strip()]
        self.add_documents(docs, chunk_size=chunk_size, overlap=overlap)
        return len(docs)

    def add_files(self, *file_paths: str | Path, chunk_size: int = 500, overlap: int = 80):
        txt_docs = []
        for fp in file_paths:
            p = Path(fp)
            if not p.is_file():
                continue
            if p.suffix.lower() == ".pdf":
                self.add_pdf_bytes(p.read_bytes(), filename=p.name, chunk_size=chunk_size, overlap=overlap)
            else:
                txt_docs.append({"source": p.name, "text": p.read_text(encoding="utf-8", errors="ignore")})
        if txt_docs:
            self.add_documents(txt_docs, chunk_size=chunk_size, overlap=overlap)

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        if self.embeddings is None or not self.chunks:
            return []
        qv = next(iter(self.embedder.embed([query]))).astype(np.float32)
        qv /= np.linalg.norm(qv) + 1e-10
        ranked = sorted(enumerate((self.embeddings @ qv).tolist()), key=lambda x: x[1], reverse=True)[:top_k]
        return [{"rank": r, "score": round(float(sv), 4), "source": self.chunks[idx]["source"], "text": self.chunks[idx]["text"]} for r, (idx, sv) in enumerate(ranked, 1)]

    def build_augmented_prompt(self, question: str, top_k: int = 3) -> tuple[str, list[dict]]:
        hits = self.search(question, top_k=top_k)
        if not hits:
            return question, []
        ctx = "\n\n".join(f"[{h['source']} | score={h['score']}]\n{h['text']}" for h in hits)
        return f"ใช้ข้อมูลอ้างอิงต่อไปนี้ตอบคำถามและระบุแหล่งที่มา ([source]):\n\n--- ข้อมูลอ้างอิง ---\n{ctx}\n-------------------\n\nคำถาม: {question}", hits

GLOBAL_RAG = StrataRAG()

# ==========================================
# 4. หน้าต่าง Studio UI (ปุ่มดาวน์โหลด Draft แยก + เปิด/ปิด MTP & N-gram)
# ==========================================
def _fmt_reply_html(raw: str, docs: list[dict] | None = None) -> str:
    out = ""
    if docs:
        items = "".join(f"<div><b style='color:#fbbf24;'>[{html_lib.escape(d['source'])}]</b> ({d['score']}): {html_lib.escape(d['text'][:180])}...</div>" for d in docs)
        out += f"<details style='background:#0f172a;border:1px solid #1e3a8a;border-radius:6px;padding:6px 10px;margin-bottom:6px;font-size:12px;color:#cbd5e1;'><summary style='cursor:pointer;color:#60a5fa;font-weight:600;'>📚 อ้างอิงจาก StrataRAG ({len(docs)} ส่วน)</summary>{items}</details>"
    thinks: list[str] = []
    vis = re.sub(r"<think>(.*?)</think>", lambda m: (thinks.append(m.group(1).strip()) or ""), raw, flags=re.DOTALL)
    if "<think>" in vis and "</think>" not in vis:
        vis, rest = vis.split("<think>", 1); thinks.append(rest.strip() + " ⏳...")
    if thinks:
        out += f"<details style='background:#0f172a;border:1px solid #334155;border-radius:6px;padding:6px 10px;margin-bottom:6px;font-size:12px;color:#94a3b8;'><summary style='cursor:pointer;color:#38bdf8;'>🧠 กระบวนการคิด</summary>{html_lib.escape(chr(10).join(thinks)).replace(chr(10), '<br>')}</details>"
    return out + f"<div style='line-height:1.6;'>{html_lib.escape(vis.strip()).replace(chr(10), '<br>') or '...'}</div>"

def _L(**kw) -> widgets.Layout:
    return widgets.Layout(**kw)

def _B(desc: str, style: str = "", w: str = "110px", **kw) -> widgets.Button:
    return widgets.Button(description=desc, button_style=style, layout=_L(width=w, **kw))

def show_model_manager_ui(model_dir: str = MODEL_DIR, port: int = DEFAULT_PORT, rag_engine: StrataRAG = GLOBAL_RAG):
    state = {"active": "ยังไม่ระบุ"}; history: list[dict] = []; btns: list[widgets.Button] = []
    header, rag_stat, spec_hint, c_stat, rag_out, a_stat = (widgets.HTML() for _ in range(6))

    def _header_html() -> str:
        cp = Path(model_dir) if Path(model_dir).exists() else Path("/content")
        u = shutil.disk_usage(cp); tg, ug, fg = u.total / (1024**3), u.used / (1024**3), u.free / (1024**3)
        pct = min(100.0, ug / tg * 100) if tg else 0.0
        bc = "#22c55e" if pct < 70 else ("#f59e0b" if pct < 88 else "#ef4444")
        hw = check_gpu_and_recommend(verbose=False); on, mid = _check_server_online(port)
        if on:
            state["active"] = mid
        sb = f"<span style='background:#14532d;color:#4ade80;padding:3px 9px;border-radius:99px;font-size:12px;'>🟢 Online (:{port} • {mid})</span>" if on else f"<span style='background:#450a0a;color:#fca5a5;padding:3px 9px;border-radius:99px;font-size:12px;'>🔴 Offline (:{port})</span>"
        spb = f"<span style='background:#3b0764;color:#e9d5ff;padding:3px 9px;border-radius:99px;font-size:12px;'>{html_lib.escape(SPEC_STATE['active_label'])}</span>" if on and not SPEC_STATE["active_label"].startswith("⏹️") else ""
        rb = f"<span style='background:#1e3a8a;color:#93c5fd;padding:3px 9px;border-radius:99px;font-size:12px;'>📚 RAG: {len(rag_engine.chunks)}</span>"
        gb = f"🎮 {hw['gpu_name']} ({hw['vram_free_gb']:.1f}/{hw['vram_total_gb']:.1f}GB)" if hw["has_gpu"] else "🖥️ CPU"
        return f"<div style='background:#0f172a;color:#f8fafc;padding:12px 16px;border-radius:10px;border:1px solid #334155;font-family:sans-serif;margin-bottom:6px;'><div style='display:flex;justify-content:space-between;flex-wrap:wrap;gap:6px;margin-bottom:6px;'><b style='font-size:15px;'>🎛️ Strata Studio (Port {port} • 10 Models + Draft/MTP/N-gram + RAG)</b><div>{sb} {spb} {rb} <span style='font-size:12px;color:#38bdf8;margin-left:4px;'>{gb}</span></div></div><div style='font-size:12px;color:#cbd5e1;margin-bottom:4px;'>💾 ดิสก์ว่าง <b style='color:{bc};'>{fg:.2f} GB</b> / {tg:.2f} GB ({pct:.1f}%)</div><div style='width:100%;background:#1e293b;height:8px;border-radius:99px;overflow:hidden;'><div style='width:{pct:.1f}%;background:{bc};height:100%;'></div></div></div>"

    def _draft_opts() -> list[tuple[str, str]]:
        opts = []
        for dk, dv in DRAFT_CATALOG.items():
            fp = _find_draft_gguf(Path(DRAFT_DIR) / dk)
            opts.append((f"✅ {dv['desc']} ({_fmt_bytes(fp.stat().st_size)})" if fp else f"⬜ {dv['desc']}", dk))
        opts.append(("⚙️ กำหนดเอง (พิมพ์ org/repo:*Q4_K_M*.gguf ช่องขวา)", "custom"))
        return opts

    spec_tb = widgets.ToggleButtons(options=[("⏹️ ปิด", "off"), ("✨ Auto", "auto"), ("🚀 MTP", "mtp"), ("⚡ Draft", "draft"), ("📚 N-gram", "ngram")], value=SPEC_STATE["mode"], description="⚡ เร่งสปีด:", style={"button_width": "82px", "description_width": "72px"})
    dmax_sl = widgets.IntSlider(value=int(SPEC_STATE["draft_max"]), min=0, max=24, description="Draft-Max:", continuous_update=False, layout=_L(width="205px"))
    chk_kvq8 = widgets.Checkbox(value=bool(SPEC_STATE["kv_q8"]), description="💾 KV Q8_0", indent=False, layout=_L(width="110px"))
    b_apply_spec = _B("🔄 รีสตาร์ทใช้โหมดนี้", "info", "150px")
    draft_dd = widgets.Dropdown(options=_draft_opts(), value="qwen35-0.8b", description="📦 Draft:", layout=_L(width="335px"))
    b_dl_draft, b_clr_draft, b_rm_draft = _B("📥 โหลด & ใช้ Draft", "primary", "145px"), _B("✖ ล้าง", "", "68px"), _B("🗑️ ลบไฟล์", "danger", "88px")
    draft_in = widgets.Text(value=str(SPEC_STATE["draft_model"]), placeholder="พาธ .gguf หรือ org/repo:*Q4_K_M*.gguf", description="ไฟล์ที่ใช้:", layout=_L(flex="1 1 250px"))

    def _upd_spec(_=None):
        SPEC_STATE.update({"mode": spec_tb.value, "draft_max": int(dmax_sl.value), "kv_q8": bool(chk_kvq8.value), "draft_model": draft_in.value.strip()})
        dm_t = "Auto (MTP=2, Draft=6, N-gram=16)" if dmax_sl.value == 0 else str(dmax_sl.value)
        df_t = Path(SPEC_STATE["draft_model"]).name if SPEC_STATE["draft_model"] and Path(SPEC_STATE["draft_model"]).is_file() else "ยังไม่ระบุ"
        mdesc = {"off": "⏹️ ปิด Speculative", "auto": "✨ Auto: Draft > MTP > N-gram", "mtp": "🚀 MTP (--spec-type draft-mtp)", "draft": f"⚡ External Draft ({html_lib.escape(df_t)})", "ngram": "📚 N-gram (--spec-type ngram-mod)"}[spec_tb.value]
        spec_hint.value = f"<span style='font-size:11.5px;color:#cbd5e1;'>{mdesc} | <b style='color:#38bdf8;'>Max:</b> {dm_t}{' | <b style=\"color:#4ade80;\">KV:Q8_0</b>' if chk_kvq8.value else ''}</span>"

    def _on_dd_chg(ch):
        fp = _find_draft_gguf(Path(DRAFT_DIR) / ch["new"]) if ch["new"] in DRAFT_CATALOG else None
        if fp:
            draft_in.value = str(fp); spec_tb.value = "mtp" if DRAFT_CATALOG[ch["new"]].get("is_mtp") else "draft"

    draft_dd.observe(_on_dd_chg, names="value")
    for w in (spec_tb, dmax_sl, chk_kvq8, draft_in):
        w.observe(_upd_spec, names="value")
    _upd_spec()

    rows_box = widgets.VBox(layout=_L(gap="6px", margin="6px 0"))
    log_out = widgets.Output(layout=_L(border="1px solid #334155", border_radius="8px", padding="8px", max_height="240px", overflow_y="auto"))
    b_all, b_cache, b_stat, b_stop, b_delall, b_ref = _B("📦 โหลดครบ 10 ตัว", "primary", "155px"), _B("🧹 ล้างแคช", "info", "105px"), _B("📡 สถานะ", "", "95px"), _B("🛑 หยุด Server", "warning", "115px"), _B("🗑️ ลบทั้งหมด", "danger", "105px"), _B("🔄 รีเฟรช", "", "88px")
    chk_del = widgets.Checkbox(value=False, description="ยืนยันลบทั้งหมด", indent=False, layout=_L(width="125px"))

    def _act(fn: Callable, *a, active: str | None = None, tab: int | None = None, **kw):
        for b in btns:
            b.disabled = True
        with log_out:
            try:
                fn(*a, **kw)
                if active:
                    state["active"] = active
                if tab is not None:
                    tabs.selected_index = tab
            except Exception as e:
                rprint(f"[bold red]❌ {e}[/bold red]")
            finally:
                _refresh()

    def _on_dl_draft(_):
        spec = draft_in.value.strip() if draft_dd.value == "custom" else draft_dd.value
        if not spec:
            spec_hint.value = "<span style='color:#fbbf24;'>⚠️ พิมพ์ org/repo:*Q4_K_M*.gguf ในช่อง 'ไฟล์ที่ใช้' ก่อนครับ</span>"; return
        def _job():
            gp, is_m = install_draft_model(spec, draft_base_dir=DRAFT_DIR)
            draft_in.value = gp; spec_tb.value = "mtp" if is_m else "draft"; _upd_spec()
        tabs.selected_index = 0; _act(_job)

    def _on_rm_draft(_):
        def _job():
            if draft_dd.value in DRAFT_CATALOG and (Path(DRAFT_DIR) / draft_dd.value).is_dir():
                shutil.rmtree(Path(DRAFT_DIR) / draft_dd.value, ignore_errors=True)
                rprint(f"[bold yellow]🗑️ ลบ Draft '{draft_dd.value}' แล้ว[/bold yellow]")
            elif draft_in.value.strip() and Path(draft_in.value.strip()).is_file():
                Path(draft_in.value.strip()).unlink(missing_ok=True)
            draft_in.value = ""
            if spec_tb.value == "draft":
                spec_tb.value = "off"
            _upd_spec()
        _act(_job)

    def _on_apply_spec(_):
        _upd_spec(); on, mid = _check_server_online(port)
        cdir = LAST_MODEL_DIR or (str(Path(model_dir) / mid) if on and (Path(model_dir) / mid).is_dir() else None)
        if cdir and Path(cdir).is_dir():
            _act(start_strata_server, model_dir=cdir, port=port, restart=True)
        else:
            spec_hint.value = "<span style='color:#fbbf24;'>✅ บันทึกโหมดแล้ว! กด 🚀 รัน & แชท ของโมเดลหลักได้เลย</span>"

    def _on_del_all(_):
        if not chk_del.value:
            with log_out:
                rprint("[yellow]⚠️ ติ๊ก 'ยืนยันลบทั้งหมด' ก่อนกดลบครับ[/yellow]")
            return
        chk_del.value = False; _act(remove_model, "all", model_dir=model_dir, stop_server=True, active="หยุดแล้ว")

    b_dl_draft.on_click(_on_dl_draft)
    b_clr_draft.on_click(lambda _: (setattr(draft_in, "value", ""), setattr(spec_tb, "value", "off" if spec_tb.value == "draft" else spec_tb.value), _upd_spec()))
    b_rm_draft.on_click(_on_rm_draft); b_apply_spec.on_click(_on_apply_spec)
    b_all.on_click(lambda _: _act(install_all_models, model_dir=model_dir, port=port))
    b_cache.on_click(lambda _: _act(clean_download_cache, model_dir=model_dir))
    b_stat.on_click(lambda _: _act(strata_server_status, port=port))
    b_stop.on_click(lambda _: _act(stop_strata_server, active="หยุดแล้ว"))
    b_delall.on_click(_on_del_all); b_ref.on_click(lambda _: _refresh())

    def _refresh():
        header.value = _header_html()
        cur_d = draft_dd.value; n_opts = _draft_opts(); draft_dd.options = n_opts
        if any(v == cur_d for _, v in n_opts):
            draft_dd.value = cur_d
        rag_stat.value = f"<b style='color:#38bdf8;'>📊 คลังความรู้ StrataRAG:</b> <b>{len(rag_engine.chunks)}</b> chunks (PDF, TXT, MD)"
        btns.clear(); btns.extend([b_all, b_cache, b_stat, b_stop, b_delall, b_ref, b_apply_spec, b_dl_draft, b_clr_draft, b_rm_draft])
        base = Path(model_dir)
        cnt_dirs = len([d for d in base.iterdir() if d.is_dir() and not d.name.startswith("_")]) if base.exists() else 0
        rws = []
        for k in ALL_PRESET_KEYS:
            cfg = MODEL_CATALOG[k]; sub = base / k; ex = sub.is_dir()
            gc = len(list(sub.rglob("*.gguf"))) if ex else 0; sz = _dir_bytes(sub) if ex else 0
            mtag = " <span style='background:#3b0764;color:#e9d5ff;padding:1px 6px;border-radius:6px;font-size:10px;'>🚀 MTP</span>" if cfg.get("has_mtp") else ""
            bdg = f"<span style='color:#4ade80;font-size:11px;'>✅ ติดตั้งแล้ว ({_fmt_bytes(sz)})</span>" if gc else f"<span style='color:#94a3b8;font-size:11px;'>⬜ ยังไม่โหลด (~{cfg['approx_gb']}GB)</span>"
            info = widgets.HTML(f"<div style='font-family:sans-serif;'><b style='color:#38bdf8;'>{k}</b>{mtag} {bdg}<div style='font-size:11px;color:#cbd5e1;'>{cfg['desc']} (<code>{cfg['repo_id']}</code>)</div></div>", layout=_L(flex="1 1 250px"))
            bd, br = _B("📥 โหลดซ้ำ" if gc else "📥 โหลด", "" if gc else "primary", "95px"), _B("🚀 รัน & แชท", "success", "110px")
            bk, bx = _B("📌 เก็บตัวเดียว", "", "105px"), _B("🗑️ ลบ", "danger", "70px")
            bk.disabled = not (ex and cnt_dirs > 1); bx.disabled = not ex
            bd.on_click(lambda _, opt=k: _act(install_model, opt, model_dir=model_dir, port=port))
            br.on_click(lambda _, opt=k, rdy=(gc > 0): _act(switch_strata_model if rdy else install_model, opt, model_dir=model_dir, port=port, active=opt, tab=1, **({} if rdy else {"auto_start_server": True})))
            bk.on_click(lambda _, opt=k: _act(keep_only_models, opt, model_dir=model_dir))
            bx.on_click(lambda _, opt=k: _act(remove_model, opt, model_dir=model_dir))
            btns.extend([bd, br, bk, bx])
            rws.append(widgets.HBox([info, widgets.HBox([bd, br, bk, bx], layout=_L(gap="5px"))], layout=_L(justify_content="space-between", align_items="center", padding="8px 10px", border="1px solid #334155", border_radius="8px", flex_flow="row wrap")))
        rows_box.children = tuple(rws)

    spec_bar = widgets.VBox([
        widgets.HBox([spec_tb, dmax_sl, chk_kvq8, b_apply_spec], layout=_L(flex_flow="row wrap", align_items="center", gap="6px")),
        widgets.HBox([draft_dd, b_dl_draft, b_clr_draft, b_rm_draft, draft_in], layout=_L(flex_flow="row wrap", align_items="center", gap="6px", width="100%")),
        spec_hint,
    ], layout=_L(padding="8px 12px", border="1px solid #334155", border_radius="10px", margin="0 0 6px 0", gap="5px"))

    tab1 = widgets.VBox([widgets.HBox([b_all, b_cache, b_stat, b_stop, chk_del, b_delall, b_ref], layout=_L(flex_flow="row wrap", gap="5px")), rows_box, log_out])

    # --- TAB 2: Chat + StrataRAG ---
    sys_in = widgets.Text(value="คุณคือผู้ช่วย AI อัจฉริยะที่ตอบชัดเจนและถูกต้อง", description="System:", layout=_L(flex="1 1 300px"))
    q_dd = widgets.Dropdown(options=[("— สลับโมเดลด่วน —", "")] + [(k, k) for k in ALL_PRESET_KEYS], value="", layout=_L(width="220px"))
    b_qsw = _B("⚡ สลับ", "warning", "90px")
    chk_rag = widgets.Checkbox(value=True, description="📚 ใช้ StrataRAG", indent=False, layout=_L(width="145px"))
    topk_sl = widgets.IntSlider(value=3, min=1, max=8, description="Top-K:", layout=_L(width="185px"))
    temp_sl = widgets.FloatSlider(value=0.7, min=0.0, max=2.0, step=0.05, description="Temp:", layout=_L(width="190px"))
    max_sl = widgets.IntSlider(value=1024, min=128, max=4096, step=64, description="MaxTok:", layout=_L(width="215px"))
    feed = widgets.VBox([widgets.HTML("<div style='color:#94a3b8;text-align:center;padding:10px;'>💬 พิมพ์ข้อความด้านล่างเพื่อเริ่มแชท</div>")], layout=_L(border="1px solid #334155", border_radius="8px", padding="10px", min_height="240px", max_height="400px", overflow_y="auto", gap="8px"))
    u_in = widgets.Textarea(placeholder="พิมพ์คำถาม...", rows=3, layout=_L(flex="1 1 auto"))
    b_send, b_clr = _B("📨 ส่งข้อความ", "success", "125px", height="66px"), _B("🗑️ ล้างแชท", "danger", "95px", height="66px")

    b_qsw.on_click(lambda _: q_dd.value and _act(switch_strata_model if (Path(model_dir) / q_dd.value).exists() else install_model, q_dd.value, model_dir=model_dir, port=port, active=q_dd.value, **({} if (Path(model_dir) / q_dd.value).exists() else {"auto_start_server": True})))
    b_clr.on_click(lambda _: (history.clear(), setattr(feed, "children", (widgets.HTML("<div style='color:#94a3b8;text-align:center;'>🧹 ล้างแชทแล้ว</div>"),))))

    def _send(_):
        q = u_in.value.strip()
        if not q:
            return
        is_on, active_mid = _check_server_online(port)
        if not is_on:
            c_stat.value = f"<span style='color:#f87171;'>❌ Server พอร์ต {port} ยังไม่เปิด! กด 🚀 รัน & แชท ที่แท็บแรกก่อนครับ</span>"; return
        u_in.value = ""; b_send.disabled = True
        aug_q, docs = rag_engine.build_augmented_prompt(q, top_k=int(topk_sl.value)) if (chk_rag.value and rag_engine.chunks) else (q, [])
        ub = widgets.HTML(f"<div style='text-align:right;'><span style='display:inline-block;background:#1d4ed8;color:#fff;padding:8px 12px;border-radius:10px;max-width:80%;text-align:left;'>👤 {html_lib.escape(q)}</span></div>")
        bb = widgets.HTML("<div style='background:#1e293b;color:#f8fafc;padding:10px;border-radius:10px;'>⏳ กำลังคิด...</div>")
        feed.children = tuple([*(feed.children if history else []), ub, bb])
        t0, buf, n = time.perf_counter(), [], 0; stag = html_lib.escape(SPEC_STATE["active_label"])
        try:
            st = get_openai_client(port=port).chat.completions.create(model=active_mid, messages=[{"role": "system", "content": sys_in.value}, *history, {"role": "user", "content": aug_q}], temperature=float(temp_sl.value), max_tokens=int(max_sl.value), stream=True)  # type: ignore[arg-type]
            for ch in st:
                c0 = next(iter(ch.choices), None); d = c0.delta.content if c0 else None
                if d:
                    buf.append(d); n += 1; spd = n / max(0.01, time.perf_counter() - t0)
                    bb.value = f"<div style='background:#1e293b;color:#f8fafc;padding:10px;border-radius:10px;border:1px solid #334155;'><div style='font-size:11px;color:#38bdf8;'>🤖 {active_mid} • <span style='color:#e9d5ff;'>{stag}</span> • <b>{spd:.1f} tok/s</b></div>{_fmt_reply_html(''.join(buf), docs)}</div>"
            el = max(0.01, time.perf_counter() - t0)
            history.extend([{"role": "user", "content": q}, {"role": "assistant", "content": "".join(buf)}])
            c_stat.value = f"<span style='color:#4ade80;font-size:12px;'>✅ เสร็จใน {el:.2f}s ({n / el:.1f} tok/s • {stag})</span>"
        except Exception as e:
            bb.value = f"<div style='background:#450a0a;color:#fecaca;padding:10px;border-radius:8px;'>❌ {html_lib.escape(str(e))}</div>"
        finally:
            b_send.disabled = False; header.value = _header_html()

    b_send.on_click(_send)
    tab2 = widgets.VBox([widgets.HBox([sys_in, q_dd, b_qsw], layout=_L(flex_flow="row wrap", gap="6px")), widgets.HBox([chk_rag, topk_sl, temp_sl, max_sl], layout=_L(flex_flow="row wrap", gap="8px")), feed, widgets.HBox([u_in, b_send, b_clr], layout=_L(gap="6px")), c_stat])

    # --- TAB 3: PDF & Text StrataRAG ---
    uploader = widgets.FileUpload(accept=".pdf,.txt,.md,.csv,.json,.py", multiple=True, description="📤 อัปโหลดไฟล์")
    b_up, b_clrrag = _B("📥 นำไฟล์เข้า RAG", "success", "165px"), _B("🗑️ ล้างคลัง RAG", "danger", "130px")
    chk_sz = widgets.IntSlider(value=500, min=200, max=1500, step=50, description="Chunk:", layout=_L(width="205px"))
    fp_in = widgets.Text(placeholder="/content/doc.pdf หรือ /content/*.pdf", description="พาธไฟล์:", layout=_L(flex="1 1 260px"))
    b_fp, b_txt, b_tq = _B("📂 โหลดจากพาธ", "info", "130px"), _B("➕ เพิ่มข้อความ", "primary", "130px"), _B("🔍 ทดสอบค้นหา", "", "125px")
    src_in = widgets.Text(value="note", description="แหล่งที่มา:", layout=_L(width="210px"))
    txt_in = widgets.Textarea(placeholder="วางข้อความ/บทความที่นี่...", rows=4, layout=_L(width="100%"))
    tq_in = widgets.Text(placeholder="คำค้นหาทดสอบ...", description="ค้นหา:", layout=_L(flex="1 1 240px"))

    def _ingest_up(_):
        items = list(uploader.value.values()) if isinstance(uploader.value, dict) else list(uploader.value)
        if not items:
            rag_out.value = "<span style='color:#fbbf24;'>⚠️ เลือกไฟล์ก่อนครับ</span>"; return
        for it in items:
            nm = it.get("name") or it.get("metadata", {}).get("name", "upload.pdf")
            raw = it.get("content", b""); rb = raw.tobytes() if hasattr(raw, "tobytes") else bytes(raw)
            rag_engine.add_pdf_bytes(rb, filename=nm, chunk_size=int(chk_sz.value)) if nm.lower().endswith(".pdf") else rag_engine.add_documents([{"source": nm, "text": rb.decode("utf-8", errors="ignore")}], chunk_size=int(chk_sz.value))
        _refresh(); rag_out.value = f"<span style='color:#4ade80;'>✅ นำเข้า {len(items)} ไฟล์สำเร็จ! ({len(rag_engine.chunks)} chunks)</span>"

    def _ingest_fp(_):
        pat = fp_in.value.strip()
        files = [Path(p) for p in (glob.glob(pat) if any(c in pat for c in "*?[]") else [pat]) if Path(p).is_file()] if pat else []
        if not files:
            rag_out.value = "<span style='color:#f87171;'>❌ ไม่พบไฟล์</span>"; return
        rag_engine.add_files(*files, chunk_size=int(chk_sz.value)); _refresh()
        rag_out.value = f"<span style='color:#4ade80;'>✅ โหลด {len(files)} ไฟล์เรียบร้อย!</span>"

    def _add_note(_):
        if txt_in.value.strip():
            rag_engine.add_documents([{"source": src_in.value or "note", "text": txt_in.value.strip()}], chunk_size=int(chk_sz.value))
            txt_in.value = ""; _refresh()

    b_up.on_click(_ingest_up); b_fp.on_click(_ingest_fp); b_txt.on_click(_add_note)
    b_clrrag.on_click(lambda _: (rag_engine.clear(), _refresh(), setattr(rag_out, "value", "<span style='color:#fbbf24;'>🧹 ล้างคลัง RAG แล้ว</span>")))
    b_tq.on_click(lambda _: setattr(rag_out, "value", "".join(f"<div style='background:#1e293b;color:#f8fafc;padding:6px 10px;border-radius:6px;margin-top:4px;font-size:12px;'><b style='color:#38bdf8;'>#{h['rank']} {html_lib.escape(h['source'])}</b> ({h['score']})<br>{html_lib.escape(h['text'])}</div>" for h in rag_engine.search(tq_in.value.strip(), top_k=int(topk_sl.value))) or "<span style='color:#fbbf24;'>ไม่พบข้อมูล</span>"))

    tab3 = widgets.VBox([rag_stat, widgets.HBox([uploader, b_up, chk_sz, b_clrrag], layout=_L(flex_flow="row wrap", gap="8px", align_items="center")), widgets.HBox([fp_in, b_fp], layout=_L(gap="6px")), widgets.HBox([src_in, b_txt], layout=_L(gap="6px")), txt_in, widgets.HBox([tq_in, b_tq], layout=_L(gap="6px")), rag_out], layout=_L(gap="6px"))

    # --- TAB 4: Arena Compare ---
    a_chks = {k: widgets.Checkbox(value=(k in ("qwen35-9b-defiant-fable", "ornith-15-9b")), description=k, indent=False, layout=_L(width="215px")) for k in ALL_PRESET_KEYS}
    a_rag = widgets.Checkbox(value=True, description="📚 แนบ StrataRAG", indent=False, layout=_L(width="155px"))
    a_dl = widgets.Checkbox(value=False, description="โหลดอัตโนมัติถ้ายังไม่มีไฟล์", indent=False, layout=_L(width="210px"))
    a_in = widgets.Textarea(placeholder="พิมพ์คำถามเดียวเพื่อเปรียบเทียบหลายโมเดล...", rows=3, layout=_L(flex="1 1 auto"))
    b_arena = _B("⚔️ เริ่มเปรียบเทียบ", "warning", "150px", height="66px")
    a_box = widgets.HBox(layout=_L(flex_flow="row wrap", gap="10px"))

    def _run_arena(_):
        _upd_spec(); q = a_in.value.strip(); sel = [k for k, c in a_chks.items() if c.value]
        if not q or not sel:
            a_stat.value = "<span style='color:#fbbf24;'>⚠️ พิมพ์คำถามและเลือกอย่างน้อย 1 โมเดล</span>"; return
        b_arena.disabled = True
        fq, docs = rag_engine.build_augmented_prompt(q, top_k=int(topk_sl.value)) if (a_rag.value and rag_engine.chunks) else (q, [])
        cards = {m: widgets.HTML(f"<div style='background:#0f172a;color:#94a3b8;padding:10px;border-radius:8px;border:1px solid #334155;'><b>🤖 {m}</b><br>⏳ รอคิว...</div>", layout=_L(flex="1 1 300px", min_width="280px")) for m in sel}
        a_box.children = tuple(cards.values()); stats = []
        try:
            for i, m in enumerate(sel, 1):
                has_f = len(list((Path(model_dir) / m).rglob("*.gguf"))) > 0 if (Path(model_dir) / m).exists() else False
                if not has_f and not a_dl.value:
                    cards[m].value = f"<div style='background:#450a0a;color:#fca5a5;padding:10px;border-radius:8px;'>⚠️ <b>{m}</b> ยังไม่ได้โหลดไฟล์</div>"; continue
                a_stat.value = f"<span style='color:#38bdf8;'>🔄 [{i}/{len(sel)}] กำลังเปิด <b>{m}</b>...</span>"
                try:
                    with log_out:
                        install_model(m, model_dir=model_dir, auto_start_server=True, port=port) if not has_f else switch_strata_model(m, model_dir=model_dir, port=port)  # type: ignore[arg-type]
                except Exception as err:
                    cards[m].value = f"<div style='background:#450a0a;color:#fecaca;padding:10px;border-radius:8px;'>❌ {html_lib.escape(str(err))}</div>"; continue
                is_ok, active_mid = _check_server_online(port)
                if not is_ok:
                    cards[m].value = "<div style='background:#450a0a;color:#fecaca;padding:10px;border-radius:8px;'>❌ Server ไม่ตอบสนอง</div>"; continue
                state["active"] = active_mid; header.value = _header_html(); stag = html_lib.escape(SPEC_STATE["active_label"])
                t0, buf, n = time.perf_counter(), [], 0
                for ch in get_openai_client(port=port).chat.completions.create(model=active_mid, messages=[{"role": "system", "content": sys_in.value}, {"role": "user", "content": fq}], temperature=float(temp_sl.value), max_tokens=512, stream=True):  # type: ignore[arg-type]
                    c0 = next(iter(ch.choices), None); d = c0.delta.content if c0 else None
                    if d:
                        buf.append(d); n += 1; spd = n / max(0.01, time.perf_counter() - t0)
                        cards[m].value = f"<div style='background:#1e293b;color:#f8fafc;padding:10px;border-radius:8px;border:1px solid #38bdf8;'><b style='color:#38bdf8;'>🤖 {m}</b> • <span style='font-size:11px;color:#e9d5ff;'>{stag}</span> ({spd:.1f} tok/s){_fmt_reply_html(''.join(buf), docs if i == 1 else None)}</div>"
                el = max(0.01, time.perf_counter() - t0); stats.append(f"{m}: {el:.1f}s ({n / el:.1f} tok/s)")
                cards[m].value = f"<div style='background:#1e293b;color:#f8fafc;padding:10px;border-radius:8px;border:1px solid #22c55e;'><b style='color:#4ade80;'>✅ {m}</b> • <span style='font-size:11px;color:#e9d5ff;'>{stag}</span> • {el:.2f}s ({n / el:.1f} tok/s){_fmt_reply_html(''.join(buf), docs if i == 1 else None)}</div>"
            a_stat.value = f"<span style='color:#4ade80;font-weight:600;'>🏆 เสร็จสิ้น: {' | '.join(stats) if stats else 'ไม่มีโมเดลที่พร้อมรัน'}</span>"
        finally:
            b_arena.disabled = False; _refresh()

    b_arena.on_click(_run_arena)
    tab4 = widgets.VBox([widgets.HBox(list(a_chks.values()), layout=_L(flex_flow="row wrap", gap="6px")), widgets.HBox([a_rag, a_dl], layout=_L(gap="12px")), widgets.HBox([a_in, b_arena], layout=_L(gap="6px")), a_stat, a_box])

    tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4])
    for idx, ttl in enumerate(["📦 1. โมเดล & ดิสก์ (10 ตัว)", "💬 2. แชท + RAG", "📚 3. คลัง PDF RAG", "⚔️ 4. เปรียบเทียบโมเดล"]):
        tabs.set_title(idx, ttl)
    _refresh()
    display(widgets.VBox([header, spec_bar, tabs]))

# ==========================================
# 5. ลงทะเบียน %%ruff + ติดตั้ง llama-server + เปิด Studio UI
# ==========================================
@register_cell_magic
def ruff(line, cell):
    no_run = "--no-run" in line.split()
    chk = subprocess.run(["ruff", "check", "--fix", "--quiet", "--config", "pyproject.toml", "--stdin-filename", "cell.py", "-"], input=cell, text=True, capture_output=True)
    fixed = chk.stdout if chk.returncode in (0, 1) and chk.stdout else cell
    fmt = subprocess.run(["ruff", "format", "--quiet", "--config", "pyproject.toml", "--stdin-filename", "cell.py", "-"], input=fixed, text=True, capture_output=True)
    code = fmt.stdout if fmt.returncode == 0 else fixed
    if code.strip() != cell.strip() or no_run:
        Console().print(Syntax(code.strip(), "python", theme="monokai", line_numbers=True))
    if not no_run:
        get_ipython().run_cell(code)

_ = install_llama_server()
print(f"✅ พร้อมใช้งานบนพอร์ต {DEFAULT_PORT} (ไม่ชนกับ Colab 8080)!")
if MODEL_OPTION != "none":
    _ = install_model(MODEL_OPTION, model_dir=MODEL_DIR, port=DEFAULT_PORT)
show_model_manager_ui(port=DEFAULT_PORT)
# === END OF SCRIPT ===

✅ พร้อมใช้งานบนพอร์ต 18080 (ไม่ชนกับ Colab 8080)!
